# Notebook 5 — Single-Head Masked Self-Attention

이제 GPT의 핵심 아이디어인 **masked self-attention**을 추가합니다.

이번 단계에서는 **single-head**에만 집중합니다.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

# ---------------------------------------------------------------------------
# [데이터 준비 파트]
# 텍스트 데이터를 다운로드하고, 컴퓨터가 이해할 수 있는 '숫자'로 변환하는 과정입니다.
# ---------------------------------------------------------------------------
if not Path("input.txt").exists():
    # 셰익스피어 대본 데이터를 다운로드합니다. (데이터 분석의 csv 다운로드와 유사합니다)
    !wget -q https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

# 텍스트 파일을 읽어옵니다.
text = open("input.txt", "r", encoding="utf-8").read()

# 고유한 문자(알파벳, 특수기호 등)만 뽑아내서 정렬합니다. (Pandas의 df['col'].unique()와 비슷함)
chars = sorted(list(set(text)))

# 문자를 숫자로(stoi), 숫자를 문자로(itos) 바꾸는 사전(Dictionary)을 만듭니다.
# 컴퓨터는 'A'라는 글자를 모르기 때문에, 'A' -> 1, 'B' -> 2 처럼 매핑해주는 것입니다.
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(chars) # 우리가 가진 고유 문자의 총 개수 (단어장 크기)

# 전체 텍스트를 숫자로 변환하여 PyTorch의 'Tensor(텐서)'로 만듭니다.
# 텐서는 NumPy의 array와 거의 똑같지만, GPU 연산이 가능하다는 장점이 있습니다.
data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

# ---------------------------------------------------------------------------
# [데이터셋 및 데이터로더 파트]
# 방대한 데이터를 한 번에 모델에 넣을 수 없으니, 작게 쪼개서(Batch) 공급하는 역할입니다.
# ---------------------------------------------------------------------------
class NextTokenDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size # 모델이 한 번에 바라볼 글자의 수 (문맥의 길이)

    def __len__(self):
        # 전체 데이터 길이에서 block_size를 뺀 만큼이 우리가 만들 수 있는 샘플의 개수입니다.
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        # x는 문제(입력), y는 정답(타겟)입니다.
        # 예: text가 "HELLO"고 block_size가 4라면
        # x = "HELL" (idx 0~3) -> y = "ELLO" (idx 1~4)
        # 즉, 모델은 'H'를 보고 'E'를, 'HE'를 보고 'L'을 맞추도록 학습됩니다.
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y

block_size = 32 # 한 번에 32개의 글자를 보고 다음 글자를 예측합니다.
dataset = NextTokenDataset(data, block_size)
# DataLoader는 데이터를 무작위로 섞어서(shuffle=True), 64개씩(batch_size) 묶어 줍니다.
loader = DataLoader(dataset, batch_size=64, shuffle=True)
xb, yb = next(iter(loader)) # xb: 문제 묶음, yb: 정답 묶음

## 1. Single-head masked self-attention

In [2]:
"""## 1. Single-head masked self-attention"""
# GPT 모델의 핵심인 '어텐션(Attention)' 메커니즘입니다.
# 문장 안에서 특정 단어가 다른 어떤 단어와 연관성이 높은지 '가중치'를 계산합니다.
class SingleHeadSelfAttention(nn.Module):
    def __init__(self, emb_dim, block_size):
        super().__init__()
        # 선형 변환(Linear)은 데이터 분석의 다중 선형 회귀(Y = WX)와 비슷합니다.
        # 각 글자가 가진 의미(Query), 각 글자가 제공할 수 있는 정보(Key), 실제 담고 있는 내용(Value)을 만듭니다.
        self.key = nn.Linear(emb_dim, emb_dim, bias=False)
        self.query = nn.Linear(emb_dim, emb_dim, bias=False)
        self.value = nn.Linear(emb_dim, emb_dim, bias=False)

        # 미래의 글자를 미리 보지 못하게 가리는 '마스크(Mask)'를 만듭니다.
        # tril은 하삼각행렬(lower triangular matrix)을 의미하며, 대각선 아래만 1이고 위는 0인 행렬입니다.
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        # x의 차원: B(Batch size: 64), T(Time/Sequence: 32), C(Channel/Embedding dim: 64)
        B, T, C = x.shape
        k = self.key(x)   # (B, T, C)
        q = self.query(x) # (B, T, C)
        v = self.value(x) # (B, T, C)

        # Query와 Key를 곱해서 '어느 글자끼리 연관성이 높은지' 점수를 매깁니다.
        # (C ** -0.5)를 곱해주는 이유는 값이 너무 커지는 것을 방지하기 위한 정규화(스케일링)입니다.
        wei = q @ k.transpose(-2, -1) * (C ** -0.5)

        # 마스킹 작업: 아직 등장하지 않은 미래의 글자 위치(0인 부분)는 점수를 -무한대(-inf)로 만듭니다.
        # 이렇게 하면 과거와 현재의 글자만 보고 다음을 예측하게 됩니다. (Causal Masking)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))

        # Softmax: 점수들을 확률(합이 1이 되도록)로 변환합니다. -무한대는 여기서 0%가 됩니다.
        wei = F.softmax(wei, dim=-1)

        # 계산된 연관성(확률 가중치)을 바탕으로 Value(실제 정보)를 섞어줍니다.
        out = wei @ v
        return out

## 2. Attention 포함 최소 모델

In [3]:
"""## 2. Attention 포함 최소 모델"""
class TinyAttentionLM(nn.Module):
    def __init__(self, vocab_size, block_size, emb_dim=64):
        super().__init__()
        # Embedding: 숫자(인덱스)를 의미를 담은 밀집 벡터(실수 배열)로 바꿉니다.
        # 예: 3 -> [0.1, -0.2, 0.5, ...]
        self.token_embedding = nn.Embedding(vocab_size, emb_dim)
        # 위치 임베딩: 글자가 문장의 몇 번째 위치에 있는지 정보를 모델에게 알려줍니다.
        self.position_embedding = nn.Embedding(block_size, emb_dim)

        self.attn = SingleHeadSelfAttention(emb_dim, block_size)

        # 최종적으로 다시 단어장(vocab_size) 크기로 차원을 변경하여,
        # 어떤 글자가 나올 확률이 가장 높은지 점수(logits)를 냅니다.
        self.lm_head = nn.Linear(emb_dim, vocab_size)

    def forward(self, x):
        B, T = x.shape
        # 0, 1, 2, ..., T-1 까지의 위치 숫자를 생성합니다.
        pos = torch.arange(T, device=x.device)

        # 글자의 의미(tok)와 글자의 위치(pos)를 더해줍니다.
        tok = self.token_embedding(x)
        pos = self.position_embedding(pos)[None]
        h = tok + pos

        # 어텐션을 통과시키며 문맥을 파악합니다.
        h = self.attn(h)
        # 최종 점수 계산
        logits = self.lm_head(h)
        return logits

# 모델을 초기화하고, 임시 데이터(xb)를 넣어 형태를 확인해봅니다.
model = TinyAttentionLM(vocab_size, block_size)
logits = model(xb)
print("logits.shape:", logits.shape) # (Batch, Time, Vocab_size) 형태가 나옵니다.

logits.shape: torch.Size([64, 32, 65])


## 3. 학습

In [4]:
"""## 3. 학습"""
# 모델이 예측한 값(logits)과 실제 정답(targets) 사이의 차이(오차)를 계산하는 함수입니다.
def sequence_cross_entropy(logits, targets):
    # Cross Entropy는 분류 문제에서 주로 쓰는 오차 함수입니다. (정답을 맞출 확률이 높을수록 오차가 0에 가까워짐)
    return F.cross_entropy(logits.transpose(1, 2), targets)

# 1 에포크(전체 데이터를 한 번 훑는 과정) 학습 함수
def train_one_epoch(model, loader, optimizer, device, max_steps=None):
    model.train() # 모델을 학습 모드로 설정 (일부 기능들이 학습에 맞게 동작하도록 함)
    total_loss, total_count = 0.0, 0

    for step, (xb, yb) in enumerate(loader):
        xb, yb = xb.to(device), yb.to(device) # 데이터를 GPU(있다면)로 보냅니다.

        logits = model(xb) # 모델에 문제를 넣고 예측값을 받습니다.
        loss = sequence_cross_entropy(logits, yb) # 정답과 비교하여 오차(Loss)를 구합니다.

        optimizer.zero_grad() # 이전 단계에서 계산된 기울기를 초기화합니다. (안 하면 누적됨)
        loss.backward()       # 오차를 바탕으로 각 가중치가 얼마나 잘못되었는지(기울기) 계산합니다. (역전파)
        optimizer.step()      # 계산된 기울기를 바탕으로 가중치를 업데이트(학습)합니다.

        # 평균 오차를 구하기 위해 기록해둡니다.
        total_loss += loss.item() * xb.size(0)
        total_count += xb.size(0)

        if max_steps is not None and step + 1 >= max_steps:
            break # 너무 오래 걸릴 수 있으니 지정된 스텝수(예: 300번)까지만 학습합니다.

    return total_loss / total_count

# 연산을 빠르게 하기 위해 GPU가 있으면 GPU를, 없으면 CPU를 사용합니다.
device = "cuda" if torch.cuda.is_available() else "cpu"
model = TinyAttentionLM(vocab_size, block_size).to(device)

# AdamW는 딥러닝에서 가장 널리 쓰이는 최적화 알고리즘 중 하나입니다. (모델 업데이트 방식을 결정)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

# 100번 반복 학습을 진행합니다. loss 값이 점점 줄어드는 것을 보는 것이 목표입니다.
for epoch in range(100):
    train_loss = train_one_epoch(model, loader, optimizer, device, max_steps=300)
    print(f"epoch {epoch:2d} | train loss {train_loss:.4f}")

epoch  0 | train loss 2.9398
epoch  1 | train loss 2.5488
epoch  2 | train loss 2.4205
epoch  3 | train loss 2.3791
epoch  4 | train loss 2.3526
epoch  5 | train loss 2.3354
epoch  6 | train loss 2.3226
epoch  7 | train loss 2.3144
epoch  8 | train loss 2.3093
epoch  9 | train loss 2.3067
epoch 10 | train loss 2.2999
epoch 11 | train loss 2.2934
epoch 12 | train loss 2.2952
epoch 13 | train loss 2.2873
epoch 14 | train loss 2.2848
epoch 15 | train loss 2.2837
epoch 16 | train loss 2.2832
epoch 17 | train loss 2.2782
epoch 18 | train loss 2.2771
epoch 19 | train loss 2.2760
epoch 20 | train loss 2.2725
epoch 21 | train loss 2.2758
epoch 22 | train loss 2.2744
epoch 23 | train loss 2.2720
epoch 24 | train loss 2.2700
epoch 25 | train loss 2.2694
epoch 26 | train loss 2.2701
epoch 27 | train loss 2.2680
epoch 28 | train loss 2.2682
epoch 29 | train loss 2.2636
epoch 30 | train loss 2.2619
epoch 31 | train loss 2.2682
epoch 32 | train loss 2.2642
epoch 33 | train loss 2.2649
epoch 34 | tra

## 4. Sampling

In [5]:
"""## 4. Sampling"""
# 학습된 모델을 사용해 실제로 글을 생성해보는 함수입니다.
@torch.no_grad() # 글을 생성할 때는 학습(기울기 계산)을 할 필요가 없으므로 꺼둡니다. (속도/메모리 절약)
def sample_attention_model(model, block_size, stoi, itos, device, start_text="ROMEO:", max_new_tokens=300):
    model.eval() # 모델을 평가/추론 모드로 변경합니다.

    # 처음에는 0으로 채워진 빈 캔버스(context)를 만듭니다.
    context = torch.zeros((1, block_size), dtype=torch.long, device=device)

    # "ROMEO:" 라는 시작 텍스트를 하나씩 숫자로 바꿔서 context에 밀어 넣습니다.
    for ch in start_text:
        if ch in stoi:
            ix = torch.tensor([[stoi[ch]]], device=device)
            # 제일 앞의 글자를 버리고, 새로운 글자를 뒤에 붙입니다. (슬라이딩 윈도우 방식)
            context = torch.cat([context[:, 1:], ix], dim=1)

    out = list(start_text)

    # 지정한 길이(max_new_tokens)만큼 새로운 글자를 하나씩 생성합니다.
    for _ in range(max_new_tokens):
        logits = model(context)
        # 문맥의 가장 마지막 글자(현재 시점)에서 예측한 다음 글자의 점수만 가져옵니다.
        logits = logits[:, -1, :]

        probs = F.softmax(logits, dim=-1) # 점수를 0~1 사이의 확률로 바꿉니다.

        # 확률에 따라 무작위로 글자를 하나 뽑습니다. (확률이 높은 글자가 뽑힐 가능성이 높음)
        # 100% 가장 높은 확률의 글자만 뽑지 않는 이유는, 생성되는 문장의 다양성을 위해서입니다.
        ix = torch.multinomial(probs, num_samples=1)

        out.append(itos[ix.item()]) # 뽑힌 숫자를 다시 글자로 바꿔서 결과에 추가합니다.

        # 방금 뽑은 글자를 다시 문맥(context)에 밀어 넣고 다음 반복을 준비합니다.
        context = torch.cat([context[:, 1:], ix], dim=1)

    return "".join(out)

# 실제로 모델이 셰익스피어 풍의 대사를 400자 생성하도록 명령합니다.
print(sample_attention_model(model, block_size, stoi, itos, device, start_text="ROMEO:", max_new_tokens=400))

ROMEO:
O bugcuns, bune bout lfor omus,
A gen, ary as you son amirke samy
Goour me lay crk but arm the,
He yem; tra!
Hand culf woulik, he-
hound! vercaws
che, blincevere
Which gs im ansinso che ter sthe; aty er:
Of:
Batred, Jheer, my, aldu aen: thotexceat ben wobl--tly.

GARLCY:
Tourd. GoHr sanigothm berat the my beda thy be. Lie to tel avest eche nod
Why shee thand I fre thavil ie Waghead schand seathin


## 5. 정리

- 각 위치는 이전 위치들을 참조할 수 있습니다.
- 미래는 causal mask로 차단됩니다.
- 이제 모델이 어떤 위치를 참고할지 스스로 결정하기 시작합니다.